In [17]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib

In [18]:
# 欄位處理 Column processing
def check_data_format(df_data, label_value):
    
    # 生活習慣&疾病史，空值刪除
    df_data = df_data.dropna(subset=["AGE","DRK","SMK_EXPERIENCE","NUT_EXPERIENCE","SPO_HABIT","ORTHOPEDICS_ARTICULUS","OSTEOPOROSIS","ARTHRITIS","GOUT","LUNG_RESPIRATORY","ASTHMA","EMPHYSEMA_BRONCHITIS","HEART_BLOOD_TUBE","VALVE_HEART_DIS","CORONARY_ARTERY_DIS","ARRHYTHMIA","CARDIOMYOPATHY","CONGENITAL_HEART_DIS","OTHER_HEART_DIS","HYPERLIPIDEMIA","HYPERTENSION","APOPLEXIA","DIABETES","ALIMENTARY_CANAL","PEPTIC_ULCER","GASTROESOPHAGEA_REFLUX","IRRITABLE_BOWEL_SND","PSYCHOSIS","DEPRESSION","MANIC_DEPRESSION","POSTPARTUM_DEPRESSION","OBSESSIVE_COMPULSIVE_DIS","ALCOHOLISM_DRUG_ABUSE","SCHIZOPHRENIA","NERVOUS_SYSTEM","EPILEPSY","HEMICRANIA","MULTIPLE_SCLEROSIS","PARKISON","DEMENTIA","LIVER_GALL_STONE","KIDNEY_STONE","RENAL_FAILURE","VERTIGO","CARCINOMA_IN_SITU","LIVER_CA","LUNG_CA","BREAST_CA","GASTRIC_CA","COLORECTAL_CA","NASOPHARYNGEAL_CA","PROSTATE_CA","OTHER_CA","CATARACT","GLAUCOMA","XEROPHTHALMIA","RENTINAL_DETACHMENT","FLOATERS","BLIND","COLOR_BLIND"])
    #print("刪除生活習慣及疾病史空值後剩餘:" + str(df_data.shape))
    
    # 身體及血尿液量測數值，空值刪除
    df_data = df_data.dropna(subset=["BODY_HEIGHT","BODY_WEIGHT","BMI","BODY_WAISTLINE","BODY_BUTTOCKS","WHR","SIT_1_SYSTOLIC_PRESSURE","SIT_1_DIASTOLIC_PRESSURE","SIT_2_SYSTOLIC_PRESSURE","SIT_2_DIASTOLIC_PRESSURE","SIT_1_HEARTBEAT_SPEED","SIT_2_HEARTBEAT_SPEED","BONE_EXAM_RESULT","T_SCORE","Z_SCORE","BODY_FAT_RATE","RBC","WBC","PLATELET","HB","HCT","HBA1C","ANTI_HCV_AB_1","HBSAG_1","HBEAG_1","ANTI_HBS_AB_1","ANTI_HBC_AB_1","FASTING_GLUCOSE","T_CHO","TG","HDL_C","LDL_C","T_BILIRUBIN","ALBUMIN","SGOT","SGPT","GAMMA_GT","AFP","BUN","CREATININE","URIC_ACID","MICROALB"])
    #print("刪除身體及血尿液量測數值空值後剩餘:" + str(df_data.shape))

    # 女性相關問題之類別資料，空值刪除
    df_data = df_data.dropna(subset=["MENSTRURATION_REGULAR","MENSTRURATION_CURR","PREGNANCY","SEVERE_DYSMENORRHEA","MYOMA","OVARIAN_CYST","ENDOMETRIOSIS","CERVICAL_POLYP","UTERINE_CA","CERVICAL_CA","OVARIAN_CA"])
    #print("刪除女性相關問題之類別資料空值後剩餘:" + str(df_data.shape))

    # 女性相關問題:初經年齡為空值，使用眾數取代
    #MENARCHE_cl = ctl_df.groupby('MENARCHE')
    #print(MENARCHE_cl.size())
    #'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead
    df_data['MENARCHE'] = df_data['MENARCHE'].fillna(df_data['MENARCHE'].mode()[0])
    #df.fillna({'MENARCHE': df_data['MENARCHE'].mode()[0]}, inplace=True)
    #print("初經年齡眾數:" + str(df_data['MENARCHE'].mode()[0]))

    # 女性相關問題:停經年齡，已停經且停經年齡為空值，使用眾數取代
    #MENOPAUSE_cl = ctl_df.groupby('MENOPAUSE')
    #print(MENOPAUSE_cl.size())
    MENOPAUSE_mode = df_data['MENOPAUSE'].mode()[0]
    df_data['MENOPAUSE'] = df_data.apply(
        lambda row: MENOPAUSE_mode if row['MENSTRURATION_CURR'] == '1' and pd.isna(row['MENOPAUSE']) else row['MENOPAUSE'],
        axis=1
    )
    #print("停經年齡眾數:" + str(MENOPAUSE_mode))

    # 女性相關問題:新增欄位"過去及至今月經期間"，(1)已停經:停經年齡-初經年齡(2)未停經:現在年齡-初經年齡
    df_data.insert(len(df_data.columns), "MC_DURATION",True)
    df_data['MC_DURATION'] = df_data.apply(
        lambda row: int(row['MENOPAUSE'])-int(row['MENARCHE']) if row['MENSTRURATION_CURR'] == '1' else int(row['AGE'])-int(row['MENARCHE']),
        axis=1
    )

    # 女性相關問題:未曾懷孕的樣本中，"是否曾經生產過"、"生產次數"、"是否曾經流產過"、"是否曾經餵過母乳"會顯示空值，需補值
    df_data['BIRTH'] = df_data.apply(
        lambda row: '1' if row['PREGNANCY'] == '1' and pd.isna(row['BIRTH']) else row['BIRTH'],
        axis=1
    )
    df_data['BIRTH_TIMES'] = df_data.apply(
        lambda row: '0' if row['BIRTH'] == '1' and pd.isna(row['BIRTH_TIMES']) else row['BIRTH_TIMES'],
        axis=1
    )

    df_data['ABORTION'] = df_data.apply(
        lambda row: '1' if row['PREGNANCY'] == '1' and pd.isna(row['ABORTION']) else row['ABORTION'],
        axis=1
    )
    df_data['BREAST_MILK'] = df_data.apply(
        lambda row: '1' if row['PREGNANCY'] == '1' and pd.isna(row['BREAST_MILK']) else row['BREAST_MILK'],
        axis=1
    )

    # 女性相關問題:"是否曾經生產過"補值後依然有空值，刪除樣本(例如:曾經懷孕，"是否曾經生產過"卻顯示空值)
    df_data = df_data.dropna(subset=["BIRTH","BIRTH_TIMES","ABORTION","BREAST_MILK"])

    # 女性相關問題:新增欄位"是否高齡產"
    # 未曾懷孕的樣本中，"最後生產年齡"為空值，補值[-1]，協助判斷未填入"最後生產年齡"的樣本以便刪除
    df_data['BIRTH_AGE_FINAL'] = df_data.apply(
        lambda row: '1' if row['PREGNANCY'] == '1' and pd.isna(row['BIRTH_AGE_FINAL']) else row['BIRTH_AGE_FINAL'],
        axis=1
    )
    df_data = df_data.dropna(subset=["BIRTH_AGE_FINAL"])
    #print("刪除女性相關問題無法補值之樣本後剩餘:" + str(df_data.shape))

    # 依"最後生產年齡"給值，>=35是高齡產，<35則否
    df_data.insert(69, "ELDERL_PRIMIGRAVIDA",True)
    df_data['ELDERL_PRIMIGRAVIDA'] = df_data.apply(
        lambda row: '1' if int(row['BIRTH_AGE_FINAL']) < 35 else '2',
        axis=1
    )

    # 身體及血尿液量測數值，特殊值處理
    # 平均收縮壓、舒張壓、心跳
    df_data.insert(91, "SIT_SYSTOLIC_PRESSURE_AVERAGE",True)
    df_data['SIT_SYSTOLIC_PRESSURE_AVERAGE'] = (df_data['SIT_1_SYSTOLIC_PRESSURE'].astype(float) + df_data['SIT_2_SYSTOLIC_PRESSURE'].astype(float))/2

    df_data.insert(92, "SIT_DIASTOLIC_PRESSURE_AVERAGE",True)
    df_data['SIT_DIASTOLIC_PRESSURE_AVERAGE'] = (df_data['SIT_1_DIASTOLIC_PRESSURE'].astype(float) + df_data['SIT_2_DIASTOLIC_PRESSURE'].astype(float))/2

    df_data.insert(93, "SIT_HEARTBEAT_SPEED_AVERAGE",True)
    df_data['SIT_HEARTBEAT_SPEED_AVERAGE'] = (df_data['SIT_1_HEARTBEAT_SPEED'].astype(float) + df_data['SIT_2_HEARTBEAT_SPEED'].astype(float))/2

    # 肺功能測量結果，文字代換成數字
    df_data['INTERPRETATION'] = df_data['INTERPRETATION'].map({'Normal Spirometry':1, 'Restrictive Impairment':2, 'Obstructive Impairment':3, 'Mixed Impairment':4})

    # 醣化血色素值，<3.8用0取代
    df_data['HBA1C'] = df_data['HBA1C'].replace(to_replace={'<3.8': 0})

    # C型肝炎抗體、B型肝炎表面抗原、B型肝炎e抗原、B型肝炎表面抗體、B型肝炎核心抗體，文字代換成數字
    df_data['ANTI_HCV_AB_1'] = df_data['ANTI_HCV_AB_1'].map({'Negative':0, 'Weakly':1, 'Positive':2, 'Equivocal':3})
    df_data['HBSAG_1'] = df_data['HBSAG_1'].map({'Negative':0, 'Weakly':1, 'Positive':2, 'Equivocal':3})
    df_data['HBEAG_1'] = df_data['HBEAG_1'].map({'Negative':0, 'Weakly':1, 'Positive':2, 'Equivocal':3})
    df_data['ANTI_HBS_AB_1'] = df_data['ANTI_HBS_AB_1'].map({'Negative':0, 'Weakly':1, 'Positive':2, 'Equivocal':3})
    df_data['ANTI_HBC_AB_1'] = df_data['ANTI_HBC_AB_1'].map({'Negative':0, 'Weakly':1, 'Positive':2, 'Equivocal':3})

    # 血清麩胺酸苯醋酸轉氨基酶，<6用0取代
    df_data['SGOT'] = df_data['SGOT'].replace(to_replace={'<6': 0})
    
    # 血清麩胺酸: 0}，<5用0取代
    df_data['SGPT'] = df_data['SGPT'].replace(to_replace={'<5': 0})

    # γ －麩胺醯轉移酶，<3.3用0取代
    df_data['GAMMA_GT'] = df_data['GAMMA_GT'].replace(to_replace={'<3.3': 0})

    # 甲型胎兒血清蛋白，<2用0取代
    df_data['AFP'] = df_data['AFP'].replace(to_replace={'< 2.00': 0, '<2': 0, '<2.0': 0, '< 2.0': 0})
    
    # 尿酸，<0.6用0取代
    df_data['URIC_ACID'] = df_data['URIC_ACID'].replace(to_replace={'<0.6': 0})

    # 尿中微蛋白，<2.09用0取代
    df_data['MICROALB'] = df_data['MICROALB'].replace(to_replace={'<2.09': 0})
    
    # 新增label
    df_data.insert(128, "OS",True)
    df_data['OS'] = label_value
    
    #print("最終樣本數:" + str(df_data.shape))

    return df_data

In [19]:
df_n = pd.read_csv ('oste_normal(20240409).csv', low_memory=False)
ctl_df = df_n[1:]
print("無骨鬆原始資料:" + str(ctl_df.shape))
#ctl_df_filter = ctl_df[pd.to_numeric(ctl_df['T_SCORE']) > 2.3]
#print(ctl_df_filter.shape)

re_ctl_df = check_data_format(ctl_df, '0')
print("無骨鬆欄位處理後資料:" + str(re_ctl_df.shape))

df_oa = pd.read_csv ('oste_osteopenia(20240409).csv', low_memory=False)
oa_df = df_oa[1:]
print("輕微骨鬆原始資料:" + str(oa_df.shape))
#oa_df_filter = oa_df[(pd.to_numeric(oa_df['T_SCORE']) > -2.0) & (pd.to_numeric(oa_df['T_SCORE']) < -1.5)]
#print(oa_df_filter.shape)

re_oa_df = check_data_format(oa_df, '2')
print("輕微骨鬆欄位處理後資料:" + str(re_oa_df.shape))

df_os = pd.read_csv ('oste_osteoporosis(20240409).csv', low_memory=False)
os_df = df_os[1:]
print("骨鬆原始資料:" + str(os_df.shape))

re_os_df = check_data_format(os_df, '1')
print("骨鬆欄位處理後資料:" + str(re_os_df.shape))

無骨鬆原始資料:(67283, 123)
無骨鬆欄位處理後資料:(59265, 129)
輕微骨鬆原始資料:(26366, 123)
輕微骨鬆欄位處理後資料:(23027, 129)
骨鬆原始資料:(7910, 123)
骨鬆欄位處理後資料:(6703, 129)


In [20]:
#合併所有分類資料
merge_all = pd.concat([re_os_df, re_ctl_df])
merge_all = pd.concat([merge_all, re_oa_df])

print(merge_all.shape)

(88995, 129)


In [21]:
#切分train data & test data
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(
    merge_all,
    test_size=0.2,
    random_state=42
)

print("train shape:", train_df.shape)
print(train_df['OS'].value_counts())
print("test shape:", test_df.shape)
print(test_df['OS'].value_counts())

train shape: (71196, 129)
OS
0    47402
2    18408
1     5386
Name: count, dtype: int64
test shape: (17799, 129)
OS
0    11863
2     4619
1     1317
Name: count, dtype: int64


In [ ]:
def sub_feature_ectract_cate(input_df):
    #category feature select
    re_cate_df = input_df.filter(items=["MENSTRURATION_CURR"
    ], axis=1)
    return re_cate_df

def sub_feature_ectract_num(input_df):
    #numeric feature select
    re_num_df = input_df.filter(items=["AGE","FASTING_GLUCOSE","SGPT","GAMMA_GT","BMI","MICROALB","RBC","BODY_WEIGHT","T_BILIRUBIN"
    ], axis=1)
    return re_num_df

In [ ]:
cate_train_df = sub_feature_ectract_cate(train_df)
cate_train_array = np.array(cate_train_df, dtype='float32')
num_train_df = sub_feature_ectract_num(train_df)

cate_test_df = sub_feature_ectract_cate(test_df)
cate_test_array = np.array(cate_test_df, dtype='float32')
num_test_df = sub_feature_ectract_num(test_df)


#僅使用train數值資料建立正規化規則，再套用到test數值資料，回傳資料型態為numpy array
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
num_train_array = scaler.fit_transform(num_train_df)

num_test_array = scaler.transform(num_test_df)

joblib.dump(scaler, "sub_feature_oste_scaler.pkl")


['sub_feature_oste_scaler.pkl']

In [24]:
#合併cate和num資料
train_array = np.concatenate([cate_train_array, num_train_array], axis=1)
train_os_array = train_df['OS'].to_numpy()

train_pos_array = train_array[train_os_array == '1']
train_neg_array = train_array[train_os_array == '0']
train_mid_array = train_array[train_os_array == '2']
print(train_pos_array.shape)
print(train_neg_array.shape)
print(train_mid_array.shape)


(5386, 10)
(47402, 10)
(18408, 10)


In [13]:
# training neg data
# 用POS取NEG

# 計算正樣本中心
median_center = np.median(train_pos_array, axis=0)
#print(median_center)
from sklearn.metrics.pairwise import euclidean_distances
# 計算負樣本到正樣本中心的距離
distances = euclidean_distances(train_neg_array, [median_center]).flatten()
#print(distances)

ext_train_neg_array = np.empty((1, train_neg_array.shape[1]), dtype=np.float32)

K = len(train_pos_array)

idx = np.argsort(distances)[-K:]
ext_train_neg_array = train_neg_array[idx]

print("POS:", train_pos_array.shape)
print("NEG:", ext_train_neg_array.shape)

POS: (5386, 10)
NEG: (5386, 10)


In [22]:
# training mid data
# 用POS和NEG取MID

ext_train_mid_array = np.empty((1, train_mid_array.shape[1]), dtype=np.float32)

# 計算正樣本中心
pos_median_center = np.median(train_pos_array, axis=0)
# 計算負樣本中心
neg_median_center = np.median(ext_train_neg_array, axis=0)

pos_distances = euclidean_distances(train_mid_array, [pos_median_center]).flatten()
neg_distances = euclidean_distances(train_mid_array, [neg_median_center]).flatten()

# 與正負樣本都保持一定距離
threshold = 2.59

mask = (
    (pos_distances >= threshold) &
    (neg_distances >= threshold)
)

ext_train_mid_array = train_mid_array[mask]

print("mid:", ext_train_mid_array.shape)

mid: (5385, 10)


In [23]:
test_array = np.concatenate([cate_test_array, num_test_array], axis=1)
test_os_array = test_df['OS'].to_numpy()

print(test_os_array)
test_pos_array = test_array[test_os_array == '1']
test_neg_array = test_array[test_os_array == '0']
test_mid_array = test_array[test_os_array == '2']
print(test_pos_array.shape)
print(test_neg_array.shape)
print(test_mid_array.shape)

['1' '0' '0' ... '0' '0' '0']
(1317, 10)
(11863, 10)
(4619, 10)


In [24]:
# testing neg data
# 用POS取NEG

# 計算正樣本中心
median_center = np.median(test_pos_array, axis=0)
#print(median_center)
from sklearn.metrics.pairwise import euclidean_distances
# 計算負樣本到正樣本中心的距離
distances = euclidean_distances(test_neg_array, [median_center]).flatten()
#print(distances)

ext_test_neg_array = np.empty((1, test_neg_array.shape[1]), dtype=np.float32)

K = len(test_pos_array)

idx = np.argsort(distances)[-K:]
ext_test_neg_array = test_neg_array[idx]

print("POS:", test_pos_array.shape)
print("NEG:", ext_test_neg_array.shape)

POS: (1317, 10)
NEG: (1317, 10)


In [27]:
# testing mid data
# 用POS和NEG取MID

ext_test_mid_array = np.empty((1, test_mid_array.shape[1]), dtype=np.float32)

# 計算正樣本中心
pos_median_center = np.median(test_pos_array, axis=0)
# 計算負樣本中心
neg_median_center = np.median(ext_test_neg_array, axis=0)

pos_distances = euclidean_distances(test_mid_array, [pos_median_center]).flatten()
neg_distances = euclidean_distances(test_mid_array, [neg_median_center]).flatten()

# 與正負樣本都保持一定距離
threshold = 2.62

mask = (
    (pos_distances >= threshold) &
    (neg_distances >= threshold)
)

ext_test_mid_array = test_mid_array[mask]

print("mid:", ext_test_mid_array.shape)

mid: (1318, 10)


In [28]:
training_X = np.vstack([train_pos_array, ext_train_neg_array])
training_X = np.vstack([training_X, ext_train_mid_array])
training_Y = np.concatenate([np.ones(train_pos_array.shape[0]), np.zeros(ext_train_neg_array.shape[0]), np.full(ext_train_mid_array.shape[0], 2)])
print(training_X.shape)
print(training_Y.shape)

testing_X = np.vstack([test_pos_array, ext_test_neg_array])
testing_X = np.vstack([testing_X, ext_test_mid_array])
testing_Y = np.concatenate([np.ones(test_pos_array.shape[0]), np.zeros(ext_test_neg_array.shape[0]), np.full(ext_test_mid_array.shape[0], 2)])
print(testing_X.shape)
print(testing_Y.shape)

(16157, 10)
(16157,)
(3952, 10)
(3952,)


In [29]:
# 儲存training&testing numpy array
np.save("sub_feature_training_X.npy", training_X)
np.save("sub_feature_training_Y.npy", training_Y)
np.save("sub_feature_testing_X.npy", testing_X)
np.save("sub_feature_testing_Y.npy", testing_Y)

In [ ]:
training_X = np.load("sub_feature_training_X.npy")
training_Y = np.load("sub_feature_training_Y.npy")
testing_X = np.load("sub_feature_testing_X.npy")
testing_Y = np.load("sub_feature_testing_Y.npy")

In [27]:
import collections
print(collections.Counter(training_Y))
print(collections.Counter(testing_Y))

Counter({np.float64(1.0): 5386, np.float64(0.0): 5386, np.float64(2.0): 5385})
Counter({np.float64(2.0): 1318, np.float64(1.0): 1317, np.float64(0.0): 1317})


In [32]:
from sklearn.metrics import accuracy_score,f1_score,matthews_corrcoef,confusion_matrix
def evalution_metrics_multi(test_label, labels_score):
    accuracy = accuracy_score(test_label, labels_score.round())
    confusion = confusion_matrix(test_label, labels_score.round())
    print(confusion)
    f1 = f1_score(test_label, labels_score.round(), average="macro")
    mcc = matthews_corrcoef(test_label, labels_score.round())
    metrics = np.round([accuracy, f1, mcc],2)
    columns=['accuracy', 'f1', 'mcc']
    metrics_dict = dict(zip(columns,metrics))
    
    return(metrics_dict)

In [33]:
from sklearn.metrics import classification_report
from sklearn.model_selection import cross_validate
import optuna

/home/wenzhi/anaconda3/envs/twb_analysis/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [25]:
cate_column = cate_train_df.columns
num_column = num_train_df.columns
feature_list = np.concatenate([cate_column, num_column])

In [35]:
from sklearn.model_selection import KFold
cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

In [28]:
from sklearn.inspection import permutation_importance
def get_permutation_importance(model_name, feature_list):
    result = permutation_importance(
        model_name, 
        testing_X, 
        testing_Y,
        scoring="f1_macro",
        n_repeats=30,
        random_state=42,
        n_jobs=-1
    ) 
    
    feature_importance = pd.DataFrame({
        "feature": feature_list,
        "importance_mean": result.importances_mean,
        "importance_std": result.importances_std
    }).sort_values("importance_mean", ascending=False)

    print(feature_importance.head(20))

In [37]:
def svm_cv(model, X_train, Y_train, cv):
    scores = cross_validate(
        model,
        X_train,
        Y_train,
        cv = cv,
        scoring = [
            'accuracy',
            'f1_macro',
            'precision_macro',
            'recall_macro',
            'matthews_corrcoef'
        ],
        n_jobs=-1
    )
    print('Accuracy:', scores['test_accuracy'])
    print('Precision:', scores['test_precision_macro'])
    print('Recall:', scores['test_recall_macro'])
    print('F1 score', scores['test_f1_macro'])
    print('matthews_corrcoef:', scores['test_matthews_corrcoef'])

    print('Accuracy:', scores['test_accuracy'].mean())
    print('Precision:', scores['test_precision_macro'].mean())
    print('Recall:', scores['test_recall_macro'].mean())
    print('F1 score', scores['test_f1_macro'].mean())
    print('matthews_corrcoef:', scores['test_matthews_corrcoef'].mean())



In [38]:
#SVM
from sklearn import svm
import joblib
from sklearn.base import clone
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    matthews_corrcoef
)
svc = svm.SVC(probability=True)
#svm_cv(svc, training_X, training_Y, cv)

# 儲存每個 fold 的結果
cv_results = {
    "accuracy": [],
    "precision_macro": [],
    "recall_macro": [],
    "f1_macro": [],
    "matthews_corrcoef": []
}

fold_models = []

for fold, (train_idx, valid_idx) in enumerate(
        cv.split(training_X, training_Y), start=1):

    print(f"Training Fold {fold}")

    # 建立新的模型
    model = clone(svc)

    # 切分資料
    X_train = training_X[train_idx]
    X_valid = training_X[valid_idx]

    y_train = training_Y[train_idx]
    y_valid = training_Y[valid_idx]

    # 訓練
    model.fit(X_train, y_train)

    # 預測
    y_pred = model.predict(X_valid)

    # 計算指標
    cv_results["accuracy"].append(
        accuracy_score(y_valid, y_pred)
    )
    cv_results["precision_macro"].append(
        precision_score(y_valid, y_pred, average="macro")
    )
    cv_results["recall_macro"].append(
        recall_score(y_valid, y_pred, average="macro")
    )
    cv_results["f1_macro"].append(
        f1_score(y_valid, y_pred, average="macro")
    )
    cv_results["matthews_corrcoef"].append(
        matthews_corrcoef(y_valid, y_pred)
    )

    # 儲存模型
    joblib.dump(model, f"sub_feature_svc_fold_{fold}.pkl")
# 顯示平均結果
print("\nCV Results")
for metric, scores in cv_results.items():
    print(
        f"{metric}: "
        f"{np.mean(scores):.4f} ± {np.std(scores):.4f}"
    )

svc_fit = svc.fit(training_X, training_Y)
svm_score = svc_fit.predict(testing_X)

print(classification_report(testing_Y.astype(float), svm_score.astype(float)))
evalution_metrics_multi(testing_Y.astype(float), svm_score.astype(float))

Training Fold 1
Training Fold 2
Training Fold 3
Training Fold 4
Training Fold 5

CV Results
accuracy: 0.7949 ± 0.0062
precision_macro: 0.8034 ± 0.0060
recall_macro: 0.7950 ± 0.0056
f1_macro: 0.7934 ± 0.0055
matthews_corrcoef: 0.6972 ± 0.0094
              precision    recall  f1-score   support

         0.0       0.78      0.92      0.84      1317
         1.0       0.91      0.74      0.82      1317
         2.0       0.71      0.72      0.72      1318

    accuracy                           0.79      3952
   macro avg       0.80      0.79      0.79      3952
weighted avg       0.80      0.79      0.79      3952

[[1209    1  107]
 [  63  969  285]
 [ 275   90  953]]


{'accuracy': np.float64(0.79), 'f1': np.float64(0.79), 'mcc': np.float64(0.69)}

In [40]:
from sklearn.ensemble import RandomForestClassifier
def rf_objective(trial, X_train, Y_train, cv):

    rf = RandomForestClassifier(
        n_estimators = trial.suggest_int("n_estimators", 10, 200),
        max_depth = trial.suggest_int("max_depth", 3, 30),
        min_samples_split = trial.suggest_int("min_samples_split", 2, 20),
        min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 10),
        max_features = trial.suggest_categorical("max_features", ["sqrt", "log2"]),
        random_state = 42,
        n_jobs = -1
    )

    scores = cross_validate(
        rf,
        X_train,
        Y_train,
        cv = cv,
        scoring = [
            'accuracy',
            'precision_macro',
            'recall_macro',
            'f1_macro',
            'matthews_corrcoef'
        ],
        n_jobs=-1
    )

    accuracy = scores["test_accuracy"].mean()
    precision = scores["test_precision_macro"].mean()
    recall = scores["test_recall_macro"].mean()
    f1 = scores["test_f1_macro"].mean()
    mcc = scores["test_matthews_corrcoef"].mean()

    # 存進 trial
    trial.set_user_attr("accuracy", accuracy)
    trial.set_user_attr("precision", precision)
    trial.set_user_attr("recall", recall)
    trial.set_user_attr("f1", f1)
    trial.set_user_attr("matthews_corrcoef", mcc)

    return f1

In [ ]:
study = optuna.create_study(direction="maximize")
study.optimize(lambda trial: rf_objective(trial, training_X, training_Y, cv), n_trials=200)

best_params = study.best_params
print(best_params)
print(f"Best F1 Macro: {study.best_value:.4f}")

best_forest = RandomForestClassifier(
    **best_params,
    random_state=42,
    n_jobs=-1
)

from sklearn.base import clone
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    matthews_corrcoef
)
import joblib
import numpy as np

# 儲存每個 fold 的結果
cv_results = {
    "accuracy": [],
    "precision_macro": [],
    "recall_macro": [],
    "f1_macro": [],
    "matthews_corrcoef": []
}

fold_models = []

for fold, (train_idx, valid_idx) in enumerate(
        cv.split(training_X, training_Y), start=1):

    print(f"Training Fold {fold}")

    # 建立新的模型
    model = clone(best_forest)

    # 切分資料
    X_train = training_X[train_idx]
    X_valid = training_X[valid_idx]

    y_train = training_Y[train_idx]
    y_valid = training_Y[valid_idx]

    # 訓練
    model.fit(X_train, y_train)

    # 預測
    y_pred = model.predict(X_valid)

    # 計算指標
    cv_results["accuracy"].append(
        accuracy_score(y_valid, y_pred)
    )
    cv_results["precision_macro"].append(
        precision_score(y_valid, y_pred, average="macro")
    )
    cv_results["recall_macro"].append(
        recall_score(y_valid, y_pred, average="macro")
    )
    cv_results["f1_macro"].append(
        f1_score(y_valid, y_pred, average="macro")
    )
    cv_results["matthews_corrcoef"].append(
        matthews_corrcoef(y_valid, y_pred)
    )

    # 儲存模型
    joblib.dump(model, f"sub_feature_rf_fold_{fold}.pkl")

    fold_models.append(model)

# 顯示平均結果
print("\nCV Results")
for metric, scores in cv_results.items():
    print(
        f"{metric}: "
        f"{np.mean(scores):.4f} ± {np.std(scores):.4f}"
    )

forest_fit = best_forest.fit(training_X, training_Y)
rf_score = forest_fit.predict(testing_X)
print(classification_report(testing_Y.astype(float), rf_score.astype(float)))
evalution_metrics_multi(testing_Y.astype(float), rf_score.astype(float))

[I 2026-06-22 15:15:01,286] A new study created in memory with name: no-name-d6ccb998-74da-44c1-b92c-c86b1d559708
[I 2026-06-22 15:15:04,516] Trial 0 finished with value: 0.7849078146281168 and parameters: {'n_estimators': 114, 'max_depth': 15, 'min_samples_split': 12, 'min_samples_leaf': 1, 'max_features': 'sqrt'}. Best is trial 0 with value: 0.7849078146281168.
[I 2026-06-22 15:15:06,798] Trial 1 finished with value: 0.7834584941933562 and parameters: {'n_estimators': 58, 'max_depth': 15, 'min_samples_split': 16, 'min_samples_leaf': 8, 'max_features': 'log2'}. Best is trial 0 with value: 0.7849078146281168.
[I 2026-06-22 15:15:09,801] Trial 2 finished with value: 0.7871050426432362 and parameters: {'n_estimators': 143, 'max_depth': 21, 'min_samples_split': 10, 'min_samples_leaf': 2, 'max_features': 'log2'}. Best is trial 2 with value: 0.7871050426432362.
[I 2026-06-22 15:15:13,283] Trial 3 finished with value: 0.786603429821066 and parameters: {'n_estimators': 198, 'max_depth': 19, '

{'n_estimators': 164, 'max_depth': 25, 'min_samples_split': 7, 'min_samples_leaf': 3, 'max_features': 'sqrt'}
Best F1 Macro: 0.7890
Training Fold 1
Training Fold 2
Training Fold 3
Training Fold 4
Training Fold 5

CV Results
accuracy: 0.7874 ± 0.0062
precision_macro: 0.7990 ± 0.0045
recall_macro: 0.7875 ± 0.0066
f1_macro: 0.7890 ± 0.0061
matthews_corrcoef: 0.6844 ± 0.0088
              precision    recall  f1-score   support

         0.0       0.81      0.85      0.83      1317
         1.0       0.90      0.74      0.81      1317
         2.0       0.67      0.76      0.71      1318

    accuracy                           0.78      3952
   macro avg       0.79      0.78      0.78      3952
weighted avg       0.79      0.78      0.78      3952

[[1118    4  195]
 [  47  970  300]
 [ 213  105 1000]]


{'accuracy': np.float64(0.78), 'f1': np.float64(0.78), 'mcc': np.float64(0.68)}

In [42]:
from xgboost import XGBClassifier

def xgb_objective(trial, X_train, Y_train, cv):
    xgb = XGBClassifier(
        n_estimators=trial.suggest_int("n_estimators", 10, 200),
        max_depth=trial.suggest_int("max_depth", 3, 10),
        learning_rate=trial.suggest_float("learning_rate", 0.001, 0.3, log=True),
        subsample=trial.suggest_float("subsample", 0.6, 1.0),
        colsample_bytree=trial.suggest_float("colsample_bytree", 0.6, 1.0),
        min_child_weight=trial.suggest_int("min_child_weight", 1, 10),
        gamma=trial.suggest_float("gamma", 0, 5),
        reg_alpha=trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
        reg_lambda=trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
        objective="multi:softprob",   # 多分類
        eval_metric="mlogloss",
        random_state=42,
        n_jobs=-1
    )
    scores = cross_validate(
        xgb,
        X_train,
        Y_train,
        cv = cv,
        scoring = [
            "accuracy",
            "precision_macro",
            "recall_macro",
            "f1_macro",
            "matthews_corrcoef"
        ],
        n_jobs=-1
    )
    
    accuracy = scores["test_accuracy"].mean()
    precision = scores["test_precision_macro"].mean()
    recall = scores["test_recall_macro"].mean()
    f1 = scores["test_f1_macro"].mean()
    mcc = scores["test_matthews_corrcoef"].mean()

    # 存進 trial
    trial.set_user_attr("accuracy", accuracy)
    trial.set_user_attr("precision", precision)
    trial.set_user_attr("recall", recall)
    trial.set_user_attr("f1", f1)
    trial.set_user_attr("matthews_corrcoef", mcc)

    return f1

In [ ]:
# 建立 XGBClassifier 模型
study = optuna.create_study(direction="maximize")
study.optimize(
    lambda trial: xgb_objective(
        trial,
        training_X,
        training_Y,
        cv
    ),
    n_trials=200
)

best_params = study.best_params
print(best_params)
print(f"Best F1 Macro: {study.best_value:.4f}")

best_xgb = XGBClassifier(
    **study.best_params,
    objective="multi:softprob",
    eval_metric="mlogloss",
    random_state=42,
    n_jobs=-1
)

from sklearn.base import clone
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    matthews_corrcoef
)
import joblib
import numpy as np

# 儲存每個 fold 的結果
cv_results = {
    "accuracy": [],
    "precision_macro": [],
    "recall_macro": [],
    "f1_macro": [],
    "matthews_corrcoef": []
}

fold_models = []

for fold, (train_idx, valid_idx) in enumerate(
        cv.split(training_X, training_Y), start=1):

    print(f"Training Fold {fold}")

    # 建立新的模型
    model = clone(best_xgb)

    # 切分資料
    X_train = training_X[train_idx]
    X_valid = training_X[valid_idx]

    y_train = training_Y[train_idx]
    y_valid = training_Y[valid_idx]

    # 訓練
    model.fit(X_train, y_train)

    # 預測
    y_pred = model.predict(X_valid)

    # 計算指標
    cv_results["accuracy"].append(
        accuracy_score(y_valid, y_pred)
    )
    cv_results["precision_macro"].append(
        precision_score(y_valid, y_pred, average="macro")
    )
    cv_results["recall_macro"].append(
        recall_score(y_valid, y_pred, average="macro")
    )
    cv_results["f1_macro"].append(
        f1_score(y_valid, y_pred, average="macro")
    )
    cv_results["matthews_corrcoef"].append(
        matthews_corrcoef(y_valid, y_pred)
    )

    # 儲存模型
    joblib.dump(model, f"sub_feature_xgb_fold_{fold}.pkl")

    fold_models.append(model)

# 顯示平均結果
print("\nCV Results")
for metric, scores in cv_results.items():
    print(
        f"{metric}: "
        f"{np.mean(scores):.4f} ± {np.std(scores):.4f}"
    )

xgb_fit = best_xgb.fit(training_X, training_Y)
xgb_score = xgb_fit.predict(testing_X)
print(classification_report(testing_Y.astype(float), xgb_score.astype(float)))
evalution_metrics_multi(testing_Y.astype(float), xgb_score.astype(float))

[I 2026-06-22 15:27:44,696] A new study created in memory with name: no-name-59cd308c-ceb6-4e26-80af-b521a9e98565
[I 2026-06-22 15:27:49,495] Trial 0 finished with value: 0.7804892615717149 and parameters: {'n_estimators': 179, 'max_depth': 8, 'learning_rate': 0.010581413162145467, 'subsample': 0.8881395290766725, 'colsample_bytree': 0.8026404112405046, 'min_child_weight': 8, 'gamma': 0.15119088085316423, 'reg_alpha': 0.0199940906423782, 'reg_lambda': 2.577086787302606e-08}. Best is trial 0 with value: 0.7804892615717149.
[I 2026-06-22 15:27:51,992] Trial 1 finished with value: 0.775789215950413 and parameters: {'n_estimators': 89, 'max_depth': 4, 'learning_rate': 0.09481421366257113, 'subsample': 0.8715423168895438, 'colsample_bytree': 0.8285067956364133, 'min_child_weight': 8, 'gamma': 1.3960447019375315, 'reg_alpha': 3.4409035721071355e-06, 'reg_lambda': 1.991026252262712e-06}. Best is trial 0 with value: 0.7804892615717149.
[I 2026-06-22 15:27:54,272] Trial 2 finished with value: 0

{'n_estimators': 91, 'max_depth': 10, 'learning_rate': 0.1033299620390487, 'subsample': 0.7879245387042029, 'colsample_bytree': 0.9533658260726684, 'min_child_weight': 10, 'gamma': 2.122112137764974, 'reg_alpha': 0.00013967038424567857, 'reg_lambda': 0.5750764950855207}
Best F1 Macro: 0.7931
Training Fold 1
Training Fold 2
Training Fold 3
Training Fold 4
Training Fold 5

CV Results
accuracy: 0.7922 ± 0.0046
precision_macro: 0.8003 ± 0.0049
recall_macro: 0.7924 ± 0.0046
f1_macro: 0.7931 ± 0.0043
matthews_corrcoef: 0.6908 ± 0.0073
              precision    recall  f1-score   support

         0.0       0.82      0.86      0.84      1317
         1.0       0.90      0.76      0.82      1317
         2.0       0.69      0.76      0.72      1318

    accuracy                           0.79      3952
   macro avg       0.80      0.79      0.80      3952
weighted avg       0.80      0.79      0.80      3952

[[1132    3  182]
 [  41  996  280]
 [ 209  102 1007]]


{'accuracy': np.float64(0.79), 'f1': np.float64(0.8), 'mcc': np.float64(0.69)}

In [44]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import cross_validate
def hgbc_objective(trial, X_train, Y_train, cv):

    hgbc = HistGradientBoostingClassifier(
        learning_rate=trial.suggest_float("learning_rate", 0.001, 0.3, log=True),
        max_iter=trial.suggest_int("max_iter", 10, 200),
        max_depth=trial.suggest_int("max_depth", 3, 20),
        max_leaf_nodes=trial.suggest_int("max_leaf_nodes", 15, 100),
        min_samples_leaf=trial.suggest_int("min_samples_leaf", 10, 100),
        l2_regularization=trial.suggest_float("l2_regularization", 1e-8, 10.0, log=True),
        max_bins=trial.suggest_int("max_bins", 64, 255),
        early_stopping=True,
        validation_fraction=0.2,
        n_iter_no_change=10,
        random_state=42
    )

    scores = cross_validate(
        hgbc,
        X_train,
        Y_train,
        cv = cv,
        scoring=[
            "accuracy",
            "precision_macro",
            "recall_macro",
            "f1_macro",
            "matthews_corrcoef"
        ],
        n_jobs=-1
    )

    accuracy = scores["test_accuracy"].mean()
    precision = scores["test_precision_macro"].mean()
    recall = scores["test_recall_macro"].mean()
    f1 = scores["test_f1_macro"].mean()
    mcc = scores["test_matthews_corrcoef"].mean()

    # 存進 trial
    trial.set_user_attr("accuracy", accuracy)
    trial.set_user_attr("precision", precision)
    trial.set_user_attr("recall", recall)
    trial.set_user_attr("f1", f1)
    trial.set_user_attr("matthews_corrcoef", mcc)

    return f1

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier
study = optuna.create_study(direction="maximize")

study.optimize(
    lambda trial: hgbc_objective(trial, training_X, training_Y, cv),
    n_trials=200
)
best_params = study.best_params
print(best_params)
print(f"Best F1 Macro: {study.best_value:.4f}")

best_hgbc = HistGradientBoostingClassifier(
    **study.best_params,
    early_stopping=True,
    validation_fraction=0.2,
    n_iter_no_change=10,
    random_state=42
)

from sklearn.base import clone
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    matthews_corrcoef
)
import joblib
import numpy as np

# 儲存每個 fold 的結果
cv_results = {
    "accuracy": [],
    "precision_macro": [],
    "recall_macro": [],
    "f1_macro": [],
    "matthews_corrcoef": []
}

fold_models = []

for fold, (train_idx, valid_idx) in enumerate(
        cv.split(training_X, training_Y), start=1):

    print(f"Training Fold {fold}")

    # 建立新的模型
    model = clone(best_hgbc)

    # 切分資料
    X_train = training_X[train_idx]
    X_valid = training_X[valid_idx]

    y_train = training_Y[train_idx]
    y_valid = training_Y[valid_idx]

    # 訓練
    model.fit(X_train, y_train)

    # 預測
    y_pred = model.predict(X_valid)

    # 計算指標
    cv_results["accuracy"].append(
        accuracy_score(y_valid, y_pred)
    )
    cv_results["precision_macro"].append(
        precision_score(y_valid, y_pred, average="macro")
    )
    cv_results["recall_macro"].append(
        recall_score(y_valid, y_pred, average="macro")
    )
    cv_results["f1_macro"].append(
        f1_score(y_valid, y_pred, average="macro")
    )
    cv_results["matthews_corrcoef"].append(
        matthews_corrcoef(y_valid, y_pred)
    )

    # 儲存模型
    joblib.dump(model, f"sub_feature_hgbc_fold_{fold}.pkl")

    fold_models.append(model)

# 顯示平均結果
print("\nCV Results")
for metric, scores in cv_results.items():
    print(
        f"{metric}: "
        f"{np.mean(scores):.4f} ± {np.std(scores):.4f}"
    )

hgbc_fit = best_hgbc.fit(training_X, training_Y)
hgbc_score = hgbc_fit.predict(testing_X)
print(classification_report(testing_Y.astype(float), hgbc_score.astype(float)))
evalution_metrics_multi(testing_Y.astype(float), hgbc_score.astype(float))

[I 2026-06-22 15:37:27,722] A new study created in memory with name: no-name-796415da-fbe1-47a0-8230-b0de7796c59e
[I 2026-06-22 15:37:32,556] Trial 0 finished with value: 0.7777959563370958 and parameters: {'learning_rate': 0.003020197548922656, 'max_iter': 170, 'max_depth': 18, 'max_leaf_nodes': 63, 'min_samples_leaf': 38, 'l2_regularization': 4.6461754404116536e-05, 'max_bins': 117}. Best is trial 0 with value: 0.7777959563370958.
[I 2026-06-22 15:37:34,858] Trial 1 finished with value: 0.7666157004113491 and parameters: {'learning_rate': 0.004585507753983412, 'max_iter': 30, 'max_depth': 19, 'max_leaf_nodes': 47, 'min_samples_leaf': 52, 'l2_regularization': 4.6398522951632904e-08, 'max_bins': 223}. Best is trial 0 with value: 0.7777959563370958.
[I 2026-06-22 15:37:37,078] Trial 2 finished with value: 0.7650454937305537 and parameters: {'learning_rate': 0.01426650983303805, 'max_iter': 28, 'max_depth': 8, 'max_leaf_nodes': 65, 'min_samples_leaf': 75, 'l2_regularization': 0.001591323

{'learning_rate': 0.04414225501195982, 'max_iter': 141, 'max_depth': 20, 'max_leaf_nodes': 19, 'min_samples_leaf': 22, 'l2_regularization': 3.410346823609005, 'max_bins': 150}
Best F1 Macro: 0.7936
Training Fold 1
Training Fold 2
Training Fold 3
Training Fold 4
Training Fold 5

CV Results
accuracy: 0.7927 ± 0.0053
precision_macro: 0.8014 ± 0.0053
recall_macro: 0.7929 ± 0.0057
f1_macro: 0.7936 ± 0.0052
matthews_corrcoef: 0.6918 ± 0.0083
              precision    recall  f1-score   support

         0.0       0.81      0.86      0.84      1317
         1.0       0.90      0.75      0.82      1317
         2.0       0.68      0.75      0.72      1318

    accuracy                           0.79      3952
   macro avg       0.80      0.79      0.79      3952
weighted avg       0.80      0.79      0.79      3952

[[1137    2  178]
 [  42  987  288]
 [ 223  102  993]]


{'accuracy': np.float64(0.79), 'f1': np.float64(0.79), 'mcc': np.float64(0.69)}

In [46]:
import joblib

joblib.dump(svc, "sub_feature_nosnp_svm(3class).pkl")
joblib.dump(best_forest, "sub_feature_nosnp_rf(3class).pkl")
joblib.dump(best_xgb, "sub_feature_nosnp_xgboost(3class).pkl")
joblib.dump(best_hgbc, "sub_feature_nosnp_HGBC(3class).pkl")

['sub_feature_nosnp_HGBC(3class).pkl']

In [ ]:
from sklearn import svm
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from xgboost import XGBClassifier
from sklearn.ensemble import HistGradientBoostingClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    matthews_corrcoef
)

# 儲存每個 fold 的結果
cv_results = {
    "accuracy": [],
    "precision_macro": [],
    "recall_macro": [],
    "f1_macro": [],
    "matthews_corrcoef": []
}

fold_models = []

for fold, (train_idx, valid_idx) in enumerate(
        cv.split(training_X, training_Y), start=1):

    print(f"Training Fold {fold}")

    clf1 = joblib.load("sub_feature_svc_fold_" + str(fold) + ".pkl")
    clf2 = joblib.load("sub_feature_rf_fold_" + str(fold) + ".pkl")
    clf3 = joblib.load("sub_feature_xgb_fold_" + str(fold) + ".pkl")
    clf4 = joblib.load("sub_feature_hgbc_fold_" + str(fold) + ".pkl")

    # 建立新的模型
    model = VotingClassifier(
        estimators=[('svm', clf1), ('rf', clf2), ('xgb', clf3), ('hgbc', clf4)],
        voting='soft'  # 'soft' for probability-based voting
    )

    # 切分資料
    X_train = training_X[train_idx]
    X_valid = training_X[valid_idx]

    y_train = training_Y[train_idx]
    y_valid = training_Y[valid_idx]

    # 訓練
    model.fit(X_train, y_train)

    # 預測
    y_pred = model.predict(X_valid)

    # 計算指標
    cv_results["accuracy"].append(
        accuracy_score(y_valid, y_pred)
    )
    cv_results["precision_macro"].append(
        precision_score(y_valid, y_pred, average="macro")
    )
    cv_results["recall_macro"].append(
        recall_score(y_valid, y_pred, average="macro")
    )
    cv_results["f1_macro"].append(
        f1_score(y_valid, y_pred, average="macro")
    )
    cv_results["matthews_corrcoef"].append(
        matthews_corrcoef(y_valid, y_pred)
    )

    # 儲存模型
    joblib.dump(model, f"sub_feature_ensemble_fold_{fold}.pkl")
# 顯示平均結果
print("\nCV Results")
for metric, scores in cv_results.items():
    print(
        f"{metric}: "
        f"{np.mean(scores):.4f} ± {np.std(scores):.4f}"
    )    

clf1 = joblib.load("sub_feature_nosnp_svm(3class).pkl")
clf2 = joblib.load("sub_feature_nosnp_rf(3class).pkl")
clf3 = joblib.load("sub_feature_nosnp_xgboost(3class).pkl")
clf4 = joblib.load("sub_feature_nosnp_HGBC(3class).pkl")
ensemble = VotingClassifier(
    estimators=[('svm', clf1), ('rf', clf2), ('xgb', clf3), ('hgbc', clf4)],
    voting='soft'  # 'soft' for probability-based voting
)

# Train
ensemble.fit(training_X, training_Y)

# Predictions
y_pred = ensemble.predict(testing_X)

# Evaluate
print(classification_report(testing_Y.astype(float), y_pred.astype(float)))
evalution_metrics_multi(testing_Y.astype(float), y_pred.astype(float))

Training Fold 1
Training Fold 2
Training Fold 3
Training Fold 4
Training Fold 5

CV Results
accuracy: 0.7933 ± 0.0052
precision_macro: 0.8014 ± 0.0049
recall_macro: 0.7934 ± 0.0055
f1_macro: 0.7937 ± 0.0050
matthews_corrcoef: 0.6928 ± 0.0080
              precision    recall  f1-score   support

         0.0       0.81      0.87      0.84      1317
         1.0       0.92      0.76      0.83      1317
         2.0       0.69      0.76      0.73      1318

    accuracy                           0.80      3952
   macro avg       0.81      0.80      0.80      3952
weighted avg       0.81      0.80      0.80      3952

[[1150    1  166]
 [  44  995  278]
 [ 220   90 1008]]


{'accuracy': np.float64(0.8), 'f1': np.float64(0.8), 'mcc': np.float64(0.7)}

In [ ]:
joblib.dump(ensemble, "sub_feature_nosnp_ensemble(3class).pkl")

In [30]:
model = joblib.load("sub_feature_nosnp_ensemble(3class).pkl")
from sklearn.inspection import permutation_importance

result = permutation_importance(
    model, 
    testing_X, 
    testing_Y,
    scoring="f1_macro",
    n_repeats=30,
    random_state=42,
    n_jobs=-1
)

feature_importance = pd.DataFrame({
    "feature": feature_list,
    "importance_mean": result.importances_mean,
    "importance_std": result.importances_std
}).sort_values("importance_mean", ascending=False)

print(feature_importance)


              feature  importance_mean  importance_std
1                 AGE         0.145962        0.006102
7                 RBC         0.079890        0.003612
8         BODY_WEIGHT         0.068952        0.003871
9         T_BILIRUBIN         0.060073        0.003350
2     FASTING_GLUCOSE         0.059492        0.003533
3                SGPT         0.057092        0.003122
4            GAMMA_GT         0.052880        0.003263
5                 BMI         0.042192        0.003410
6            MICROALB         0.030707        0.002788
0  MENSTRURATION_CURR         0.016438        0.003383
